# Modelo Baseline — Random Forest (sklearn)
PySpark para leer y procesar datos desde Delta — sklearn para el entrenamiento  
Shuffle aleatorio + split estratificado 80/20 — métricas para datos desbalanceados

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt

DELTA_FE_PATH = "/Volumes/workspace/default/network_data/features_fe/"
df = spark.read.format("delta").load(DELTA_FE_PATH)
print(f"Total ventanas : {df.count():,}")
print(f"Columnas       : {len(df.columns)}")

## 1 — Seleccionar features y pasar a pandas

In [0]:
exclude = ["window_id", "window_start", "window_end", "session_id", "label"]

feature_cols = [
    c for c in df.columns
    if c not in exclude
    and df.schema[c].dataType.simpleString() in ("double", "long", "int", "integer", "float")
]

print(f"Features para el modelo: {len(feature_cols)}")
print(feature_cols)

In [0]:
# Cast a double, rellenar nulls y traer a pandas
# NO ordenamos temporalmente — queremos el orden original para luego shufflear
pdf = (
    df.select(feature_cols + ["label"])
      .toPandas()
)

pdf[feature_cols] = pdf[feature_cols].fillna(0).astype(float)
pdf["label"]      = pdf["label"].astype(int)

print(f"Shape      : {pdf.shape}")
print(f"Nulls      : {pdf[feature_cols].isnull().sum().sum()}")
print(f"Normal (0) : {(pdf['label']==0).sum():,}")
print(f"Ataque (1) : {(pdf['label']==1).sum():,}")

## 2 — Shuffle + Split estratificado 80/20

In [0]:
X = pdf[feature_cols].values
y = pdf["label"].values

# Shuffle aleatorio + split estratificado
# - shuffle=True mezcla los datos aleatoriamente antes de dividir
# - stratify=y garantiza que train y test tengan la misma proporción 94/6
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=42
)

print(f"Train : {len(X_train):,}  |  Ataques: {y_train.sum():,}  ({y_train.mean()*100:.2f}%)")
print(f"Test  : {len(X_test):,}   |  Ataques: {y_test.sum():,}  ({y_test.mean()*100:.2f}%)")

# Peso de clase para compensar desbalanceo
n_normal = (y_train == 0).sum()
n_ataque = (y_train == 1).sum()
weight_ataque = round(n_normal / n_ataque, 2)
print(f"\nPeso clase Normal (0) : 1.0")
print(f"Peso clase Ataque (1) : {weight_ataque}")

## 3 — Entrenar Random Forest

In [0]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    class_weight={0: 1.0, 1: weight_ataque},
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Entrenando Random Forest (100 árboles, max_depth=10)...")
rf.fit(X_train, y_train)
print("✅ Entrenamiento completado")

## 4 — Evaluación

In [0]:
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print("=" * 50)
print("MÉTRICAS DE EVALUACIÓN")
print("=" * 50)
print(classification_report(
    y_test, y_pred,
    target_names=["Normal (0)", "Ataque (1)"]
))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"AUC-ROC                : {roc_auc_score(y_test, y_proba):.4f}")
print(f"AUC-PR                 : {average_precision_score(y_test, y_proba):.4f}")
print("-" * 50)
print(f"Verdaderos Negativos   : {tn:,}  — Normales correctos")
print(f"Falsos Positivos       : {fp:,}  — Normales como Ataque")
print(f"Falsos Negativos       : {fn:,}  — Ataques no detectados")
print(f"Verdaderos Positivos   : {tp:,}  — Ataques detectados")
print("-" * 50)
print(f"Tasa detección ataques : {tp/(tp+fn)*100:.2f}%")
print(f"Tasa falsa alarma      : {fp/(fp+tn)*100:.2f}%")
print("=" * 50)

In [0]:
# Matriz de confusión + Curva ROC
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusión
im = axes[0].imshow(cm, cmap="Blues")
plt.colorbar(im, ax=axes[0])
labels_cm = ["Normal (0)", "Ataque (1)"]
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(labels_cm)
axes[0].set_yticklabels(labels_cm)
axes[0].set_xlabel("Predicción", fontsize=12)
axes[0].set_ylabel("Real", fontsize=12)
axes[0].set_title("Matriz de Confusión", fontsize=13, fontweight="bold")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{cm[i,j]:,}",
                     ha="center", va="center",
                     color="white" if cm[i,j] > cm.max()/2 else "black",
                     fontsize=14, fontweight="bold")

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_val = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color="#4C8BF5", lw=2, label=f"AUC = {roc_val:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("Tasa Falsos Positivos")
axes[1].set_ylabel("Tasa Verdaderos Positivos")
axes[1].set_title("Curva ROC", fontsize=13, fontweight="bold")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Curva Precision-Recall
prec, rec, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rec, prec, color="#E8453C", lw=2, label=f"AP = {ap:.4f}")
ax.axhline(y=y_test.mean(), color="gray", linestyle="--", lw=1,
           label=f"Baseline ({y_test.mean():.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Curva Precision-Recall", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5 — Importancia de features

In [0]:
importances = pd.DataFrame({
    "feature":    feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Top 20 features más importantes:")
print(importances.head(20).to_string(index=False))

top20 = importances.head(20)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20["feature"][::-1], top20["importance"][::-1],
        color="#4C8BF5", edgecolor="white")
ax.axvline(importances["importance"].mean(), color="gray",
           linestyle="--", lw=0.8,
           label=f"Media ({importances['importance'].mean():.4f})")
ax.set_title("Top 20 features — Importancia Random Forest",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Importancia (Gini)")
ax.legend()
plt.tight_layout()
plt.show()

below_mean = importances[importances["importance"] < importances["importance"].mean()]
print(f"\nFeatures por debajo de la media: {len(below_mean)}")
print(below_mean["feature"].tolist())

## 6 — Ajuste de umbral de clasificación

In [0]:
thresholds_range = np.arange(0.1, 0.9, 0.05)
results = []

for t in thresholds_range:
    y_pred_t = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    results.append({
        "threshold":        round(t, 2),
        "recall_ataque":    round(tp_t / (tp_t + fn_t) * 100, 2),
        "precision_ataque": round(tp_t / (tp_t + fp_t) * 100, 2) if (tp_t + fp_t) > 0 else 0,
        "falsa_alarma":     round(fp_t / (fp_t + tn_t) * 100, 2),
        "f1_ataque":        round(2 * tp_t / (2 * tp_t + fp_t + fn_t) * 100, 2) if (2*tp_t + fp_t + fn_t) > 0 else 0
    })

df_thresh = pd.DataFrame(results)
print(df_thresh.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_thresh["threshold"], df_thresh["recall_ataque"],
        color="#E8453C", lw=2, label="Recall Ataque (%)")
ax.plot(df_thresh["threshold"], df_thresh["precision_ataque"],
        color="#4C8BF5", lw=2, label="Precision Ataque (%)")
ax.plot(df_thresh["threshold"], df_thresh["falsa_alarma"],
        color="gray", lw=1.5, linestyle="--", label="Falsa Alarma (%)")
ax.plot(df_thresh["threshold"], df_thresh["f1_ataque"],
        color="#F59E0B", lw=2, label="F1 Ataque (%)")
ax.axvline(0.5, color="black", linestyle=":", lw=1, label="Umbral default (0.5)")
ax.set_xlabel("Umbral de clasificación")
ax.set_ylabel("%")
ax.set_title("Métricas vs Umbral de clasificación", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best = df_thresh.loc[df_thresh["f1_ataque"].idxmax()]
print(f"\nUmbral óptimo por F1  : {best['threshold']}")
print(f"  Recall Ataque       : {best['recall_ataque']}%")
print(f"  Precision Ataque    : {best['precision_ataque']}%")
print(f"  Falsa Alarma        : {best['falsa_alarma']}%")
print(f"  F1 Ataque           : {best['f1_ataque']}%")

## 7 — Resumen final

In [0]:
print("=" * 55)
print("RESUMEN MODELO — RANDOM FOREST (shuffle + estratificado)")
print("=" * 55)
print(f"  Delta usado          : features_fe2")
print(f"  Features usadas      : {len(feature_cols)}")
print(f"  Ventanas train       : {len(X_train):,}")
print(f"  Ventanas test        : {len(X_test):,}")
print(f"  Peso clase ataque    : {weight_ataque}")
print("-" * 55)
print(f"  AUC-ROC              : {roc_auc_score(y_test, y_proba):.4f}")
print(f"  AUC-PR               : {average_precision_score(y_test, y_proba):.4f}")
print(f"  Tasa detección att   : {tp/(tp+fn)*100:.2f}%  (umbral 0.5)")
print(f"  Tasa falsa alarma    : {fp/(fp+tn)*100:.2f}%  (umbral 0.5)")
print("-" * 55)
print(f"  Umbral óptimo (F1)   : {best['threshold']}")
print(f"  Recall con óptimo    : {best['recall_ataque']}%")
print(f"  Falsa alarma óptimo  : {best['falsa_alarma']}%")
print("-" * 55)
print(f"  Top feature          : {importances.iloc[0]['feature']}")
print(f"  Importancia top      : {importances.iloc[0]['importance']:.4f}")
print("=" * 55)